# Notebook 5: MLflow Model Registry

**Learning Objectives:**
- Register models in MLflow Model Registry
- Manage model versions
- Transition models through stages (Staging → Production)
- Load and use registered models
- Update model metadata and descriptions

**Dataset:** Wine classification

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import time

## Step 1: Prepare Data

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset: {X.shape}")
print(f"Classes: {wine.target_names}")

## Step 2: Train and Register First Model

In [ ]:
mlflow.set_experiment("05_Model_Registry")

with mlflow.start_run(run_name="Wine_Classifier_v1") as run:
    
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Log parameters and metrics
    mlflow.log_params({"n_estimators": 100, "max_depth": 5})
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    # Register model in Model Registry
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="WineClassifier"
    )
    
    run_id = run.info.run_id
    
    print(f" Model registered: WineClassifier")
    print(f"   Version: 1")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1 Score: {f1:.4f}")
    print(f"   Run ID: {run_id}")

## Step 3: Train and Register Second Model Version

In [ ]:
with mlflow.start_run(run_name="Wine_Classifier_v2"):
    
    model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    mlflow.log_params({"n_estimators": 200, "max_depth": 10})
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="WineClassifier"
    )
    
    print(f" New model version registered: WineClassifier")
    print(f"   Version: 2")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1 Score: {f1:.4f}")

## Step 4: Train Different Algorithm (Version 3)

In [ ]:
with mlflow.start_run(run_name="Wine_Classifier_v3_GB"):
    
    model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    mlflow.log_params({"algorithm": "GradientBoosting", "n_estimators": 100, "learning_rate": 0.1})
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    # Register as version 3
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="WineClassifier"
    )
    
    print(f" Third model version registered: WineClassifier")
    print(f"   Version: 3 (Gradient Boosting)")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1 Score: {f1:.4f}")

## Step 5: Manage Model Versions with MlflowClient

In [ ]:
client = MlflowClient()

model_name = "WineClassifier"
versions = client.search_model_versions(f"name='{model_name}'")

print("\n" + "="*70)
print(f"ALL VERSIONS OF '{model_name}'")
print("="*70)

for version in versions:
    print(f"\nVersion: {version.version}")
    print(f"  Stage: {version.current_stage}")
    print(f"  Run ID: {version.run_id}")
    print(f"  Created: {version.creation_timestamp}")

## Step 6: Transition Models Through Stages

In [ ]:
# Promote version 1 to Staging
client.transition_model_version_stage(
    name="WineClassifier",
    version=1,
    stage="Staging"
)
print("✅ Version 1 → Staging")

# Promote version 2 to Production
client.transition_model_version_stage(
    name="WineClassifier",
    version=2,
    stage="Production"
)
print("✅ Version 2 → Production")

# Keep version 3 in None (testing)
print("ℹ️  Version 3 → None (under evaluation)")

print("\n💡 Model lifecycle:")
print("   None → Staging → Production → Archived")

## Step 7: Update Model Descriptions

In [ ]:
client.update_model_version(
    name="WineClassifier",
    version=1,
    description="Baseline RandomForest model with 100 trees, max_depth=5. Good starting point."
)

client.update_model_version(
    name="WineClassifier",
    version=2,
    description="Improved RandomForest with 200 trees, max_depth=10. Current production model."
)

client.update_model_version(
    name="WineClassifier",
    version=3,
    description="Experimental GradientBoosting model. Under evaluation for production."
)

print(" Descriptions updated for all versions")

## Step 8: Load and Use Registered Models

In [ ]:
# Load specific version
model_v1 = mlflow.pyfunc.load_model(model_uri="models:/WineClassifier/1")
print(" Loaded Version 1")

# Load model from specific stage
production_model = mlflow.pyfunc.load_model(model_uri="models:/WineClassifier/Production")
print(" Loaded Production model")

staging_model = mlflow.pyfunc.load_model(model_uri="models:/WineClassifier/Staging")
print(" Loaded Staging model")

predictions = production_model.predict(X_test[:5])
print(f"\n📊 Production model predictions (first 5 samples):")
print(predictions)
print(f"\nTrue labels:")
print(y_test[:5])

## Step 9: Compare Model Versions

In [ ]:
# Get metrics for all versions
results = []

for version in [1, 2, 3]:
    model = mlflow.pyfunc.load_model(model_uri=f"models:/WineClassifier/{version}")
    
    version_info = client.get_model_version("WineClassifier", version)
    
    run = client.get_run(version_info.run_id)
    
    results.append({
        "version": version,
        "stage": version_info.current_stage,
        "accuracy": run.data.metrics.get("accuracy", 0),
        "f1_score": run.data.metrics.get("f1_score", 0)
    })

df_comparison = pd.DataFrame(results)
print("\n" + "="*70)
print("MODEL VERSION COMPARISON")
print("="*70)
print(df_comparison.to_string(index=False))

print("\n Version 2 is in Production stage with best performance!")

## Step 10: Archive Old Version

In [ ]:
client.transition_model_version_stage(
    name="WineClassifier",
    version=1,
    stage="Archived"
)

print(" Version 1 → Archived")
print("\n Current model lifecycle:")
print("   Version 1: Archived (old baseline)")
print("   Version 2: Production (current best)")
print("   Version 3: None (under evaluation)")

## 🎯 Key Takeaways

1. **Model Registry** - Centralized model management
2. **Versioning** - Automatic version tracking
3. **Stages** - None → Staging → Production → Archived
4. **Loading Models** - By version or by stage
5. **Metadata** - Descriptions, tags for documentation
6. **Comparison** - Easy comparison between versions

## 📝 Exercise

1. Go to MLflow UI → Models tab
2. Click on "WineClassifier"
3. View all 3 versions and their stages
4. Try promoting Version 3 to Staging
5. Compare metrics across versions

## 🚀 Production Workflow

```python
# In production code, always use stage name:
model = mlflow.pyfunc.load_model("models:/WineClassifier/Production")
predictions = model.predict(new_data)
```

This ensures you're always using the approved production model!